# Real magnetization snapshots along the hysteresis loop

This notebook is intentionally separate from the production simulation notebooks. It reruns one selected FePt cap and saves real spatial magnetization states from `system.m` at key points of the loop.

Outputs are written to a timestamped folder and include:

- per-state PNG side-view maps of `m_z/M_s`
- per-state NPZ arrays for later replotting
- a CSV hysteresis loop
- a slide-style summary figure with the loop and selected snapshots


In [ ]:
# Imports
import os
import glob
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

import plotly.graph_objects as go
from plotly.subplots import make_subplots

import micromagneticmodel as mm
import discretisedfield as df
import oommfc as oc

mu0 = mm.consts.mu0



## User controls

Start with one representative cap. A 10 um cap with 15% A1 is the closest default to the example slide, but you can change these values before running.

In [ ]:
# Geometry and phase choice
SPHERE_DIAMETER_UM = 10.0
CAP_THICKNESS_NM = 60.0
SOFT_FRACTION_A1 = 0.15  # 0.15 = 15% A1; set 0.0 for pure L10 behavior
ANISOTROPY_MODE = "Radial"  # "Radial" or "Uniaxial_Vertical"

# Solver choice
T = 0  # 0 K uses MinDriver; T > 0 uses TimeDriver with thermal field
RUN_TIME = 1e-10
ALPHA = 0.05

# Material parameters
Ms = 1.0e6      # A/m
A = 1.0e-11     # J/m
Ku_hard = 6.6e6 # J/m^3, L10 phase
Ku_soft = 1.0e4 # J/m^3, A1 phase

# Field sweep
B_MAX = 18.0
N_PER_BRANCH = 41  # 41 down + 41 up = 82 total states for a lighter snapshot dataset
B_TILT = 0.01

# Reproducibility for A1/L10 spatial assignment
RANDOM_SEED = 42

# Mesh: matches the conservative strategy used in the production notebooks
N_FIXED = 110
SMALL_CAP_CELL_SIZE = 18e-9

# Snapshot output controls
SAVE_PNG_EVERY_STATE = False  # keep False for speed; render selected PNG/SVG at the end

# Output
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
soft_pct = int(round(SOFT_FRACTION_A1 * 100))
T_label = f"{T}K"
RUN_LABEL = f"cap_{SPHERE_DIAMETER_UM:g}um_{ANISOTROPY_MODE}_{soft_pct}pct_A1_{T_label}_{2 * N_PER_BRANCH}states"
OUTPUT_FOLDER = f"Snapshot_Study_{RUN_LABEL}_{timestamp}"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

print(f"Output folder: {OUTPUT_FOLDER}")


## Helper functions

The simulation cell saves one `.npz` file per field step. Each file contains real spatial projections from `system.m`: an XZ side view and an XY top view of `m_z/M_s`, averaged only where magnetic material exists. White/empty pixels represent no FePt material along that projected line of sight.


In [ ]:
def build_field_schedule(B_max=B_MAX, n_per_branch=N_PER_BRANCH):
    return np.concatenate([
        np.linspace(B_max, -B_max, n_per_branch),
        np.linspace(-B_max, B_max, n_per_branch),
    ])


def branch_label(step_index, total_steps):
    return "descending" if step_index < total_steps / 2 else "ascending"


def mesh_axis(region, axis, n):
    pmin = np.asarray(region.pmin, dtype=float)
    pmax = np.asarray(region.pmax, dtype=float)
    step = (pmax[axis] - pmin[axis]) / n
    return pmin[axis] + step * (np.arange(n) + 0.5)


def material_average(values, mask, axis):
    weighted_sum = np.where(mask, values, 0.0).sum(axis=axis)
    counts = mask.sum(axis=axis)
    return np.divide(
        weighted_sum,
        counts,
        out=np.full_like(weighted_sum, np.nan, dtype=float),
        where=counts > 0,
    )


def state_projections_mz(field, Ms_value):
    arr = np.asarray(field.array)
    mz = arr[..., 2] / Ms_value
    magnetic = np.linalg.norm(arr, axis=-1) > (0.05 * Ms_value)

    nx, ny, nz = arr.shape[:3]
    x = mesh_axis(field.mesh.region, 0, nx) * 1e6
    y = mesh_axis(field.mesh.region, 1, ny) * 1e6
    z = mesh_axis(field.mesh.region, 2, nz) * 1e6

    # XZ side view: average through y. XY top view: average through z.
    xz = material_average(mz, magnetic, axis=1).T
    xy = material_average(mz, magnetic, axis=2).T
    return x, y, z, xz, xy


def render_xz_state(npz_path, outdir=None, basename=None, show=True):
    data = np.load(npz_path, allow_pickle=True)
    x_um = data["x_um"]
    z_um = data["z_um"]
    mz_xz = data["mz_over_Ms"]
    label = str(data["label"])
    B_z = float(data["B_ext_T"])
    mz_avg = float(data["Mz_over_Ms"])
    step_index = int(data["step_index"])

    if outdir is None:
        outdir = os.path.dirname(npz_path) or "."
    if basename is None:
        basename = os.path.splitext(os.path.basename(npz_path))[0]

    png_path = os.path.join(outdir, basename + ".png")
    svg_path = os.path.join(outdir, basename + ".svg")

    fig, ax = plt.subplots(figsize=(5.6, 3.8))
    image = ax.imshow(
        mz_xz,
        origin="lower",
        extent=[x_um.min(), x_um.max(), z_um.min(), z_um.max()],
        cmap="coolwarm",
        vmin=-1,
        vmax=1,
        interpolation="nearest",
        aspect="equal",
    )
    ax.set_title(f"{label} | step {step_index}\nB = {B_z:+.2f} T, Mz/Ms = {mz_avg:+.3f}", fontsize=10)
    ax.set_xlabel("x (um)")
    ax.set_ylabel("z (um)")
    cbar = fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("m_z / M_s")
    fig.tight_layout()
    fig.savefig(png_path, dpi=300)
    fig.savefig(svg_path)
    if show:
        plt.show()
    else:
        plt.close(fig)
    return png_path, svg_path


def save_xz_state(field, label, B_z, mz_avg, step_index, outdir=OUTPUT_FOLDER, save_png=False):
    x_um, y_um, z_um, mz_xz, mz_xy = state_projections_mz(field, Ms)
    safe_label = label.lower().replace(" ", "_").replace("/", "_")
    stem = f"state_{step_index:03d}_{safe_label}_B{B_z:+.3f}T"
    npz_path = os.path.join(outdir, stem + ".npz")

    np.savez_compressed(
        npz_path,
        x_um=x_um,
        y_um=y_um,
        z_um=z_um,
        mz_over_Ms=mz_xz,
        xy_mz_over_Ms=mz_xy,
        B_ext_T=B_z,
        Mz_over_Ms=mz_avg,
        step_index=step_index,
        label=label,
        branch=branch_label(step_index, len(B_fields)) if "B_fields" in globals() else "unknown",
    )

    png_path = ""
    svg_path = ""
    if save_png:
        png_path, svg_path = render_xz_state(npz_path, show=False)

    return {
        "step_index": step_index,
        "label": label,
        "branch": branch_label(step_index, len(B_fields)) if "B_fields" in globals() else "unknown",
        "B_ext_T": B_z,
        "Mz_over_Ms": mz_avg,
        "npz_path": npz_path,
        "png_path": png_path,
        "svg_path": svg_path,
    }


def find_latest_omf(dirname, drive_number):
    drive_dir = os.path.join(dirname, f"drive-{drive_number}")
    candidates = glob.glob(os.path.join(drive_dir, "*.omf"))
    candidates = [p for p in candidates if os.path.basename(p).lower() != "m0.omf"] or candidates
    if not candidates:
        candidates = glob.glob(os.path.join(dirname, "**", "*.omf"), recursive=True)
        candidates = [p for p in candidates if os.path.basename(p).lower() != "m0.omf"] or candidates
    return max(candidates, key=os.path.getmtime) if candidates else None


def drive_and_refresh_m(system, driver, dirname, **drive_kwargs):
    drive_number = system.drive_number
    try:
        driver.drive(system, dirname=dirname, **drive_kwargs)
    except IndexError as exc:
        omf_path = find_latest_omf(dirname, drive_number)
        if omf_path is None:
            raise RuntimeError(
                f"OOMMF finished drive-{drive_number}, but no .omf file was found in {dirname}. "
                "Check the OOMMF log in the run directory."
            ) from exc

        system.m.array = df.Field.from_file(omf_path).array
        system.drive_number += 1
        print(f"Recovered magnetization from {omf_path}")


## Build and run one real micromagnetic loop

This lighter run saves all 82 states by default. The output is a manifest plus one `.npz` file per field step, so you can render any state later without rerunning OOMMF.


In [ ]:
np.random.seed(RANDOM_SEED)

sphere_diameter = SPHERE_DIAMETER_UM * 1e-6
cap_thickness = CAP_THICKNESS_NM * 1e-9
R_inner = sphere_diameter / 2
R_outer = R_inner + cap_thickness

if sphere_diameter <= 1.5e-6:
    n_val = int(np.ceil((2 * R_outer) / SMALL_CAP_CELL_SIZE))
else:
    n_val = N_FIXED

region = df.Region(p1=(-R_outer, -R_outer, 0), p2=(R_outer, R_outer, R_outer))
mesh = df.Mesh(region=region, n=(n_val, n_val, n_val))

print(f"Mesh: {n_val} x {n_val} x {n_val}")


def Ms_mask(pos):
    x, y, z = pos
    r = np.sqrt(x*x + y*y + z*z)
    return Ms if (R_inner <= r <= R_outer and z >= 0) else 0


def Ku_distribution(pos):
    x, y, z = pos
    r = np.sqrt(x*x + y*y + z*z)
    if not (R_inner <= r <= R_outer and z >= 0):
        return 0
    return Ku_soft if np.random.random() < SOFT_FRACTION_A1 else Ku_hard


def u_func(pos):
    x, y, z = pos
    if ANISOTROPY_MODE == "Uniaxial_Vertical":
        return (0, 0, 1)

    r = np.sqrt(x*x + y*y + z*z)
    return (x/r, y/r, z/r) if r > 0 else (0, 0, 1)


system = mm.System(name=RUN_LABEL)
system.energy = (
    mm.Exchange(A=A)
    + mm.UniaxialAnisotropy(
        K=df.Field(mesh, nvdim=1, value=Ku_distribution),
        u=df.Field(mesh, nvdim=3, value=u_func),
    )
    + mm.Demag()
    + mm.Zeeman(H=(0, 0, 0))
)

if T > 0:
    system.dynamics = mm.Precession(gamma0=mm.consts.gamma0) + mm.Damping(alpha=ALPHA)

system.m = df.Field(mesh, nvdim=3, value=(0, 0, 1), norm=df.Field(mesh, nvdim=1, value=Ms_mask))

v_theory = (2/3) * np.pi * (R_outer**3 - R_inner**3)
v_mesh = (system.m.norm.integrate() / Ms).item()
print(f"Volume error: {abs(v_mesh - v_theory) / v_theory * 100:.2f}%")


In [ ]:
B_fields = build_field_schedule()
driver = oc.TimeDriver() if T > 0 else oc.MinDriver()

hyst_data = []
snapshot_records = []
previous_mz = None

for s_idx, B_z in enumerate(B_fields):
    system.energy.zeeman.H = (0, B_TILT / mu0, B_z / mu0)

    if T > 0:
        drive_and_refresh_m(system, driver, RUN_LABEL, t=RUN_TIME, n=1, T=T, n_threads=8)
    else:
        drive_and_refresh_m(system, driver, RUN_LABEL, n_threads=8, stopping_mxHxm=2e4, total_iteration_limit=15000)

    m_norm_int = system.m.norm.integrate().item()
    mz_avg = (system.m.z.integrate().item() / m_norm_int) if m_norm_int > 0 else 0.0
    hyst_data.append(mz_avg)

    label = f"{branch_label(s_idx, len(B_fields)).capitalize()} state"
    snapshot_records.append(save_xz_state(system.m, label, B_z, mz_avg, s_idx, save_png=SAVE_PNG_EVERY_STATE))

    previous_mz = mz_avg

    if s_idx % 10 == 0:
        print(f"Step {s_idx + 1:3d}/{len(B_fields)} | B = {B_z:+6.2f} T | Mz/Ms = {mz_avg:+.4f}")

loop_df = pd.DataFrame({
    "step_index": np.arange(len(B_fields)),
    "B_ext_T": B_fields,
    "Mz_over_Ms": hyst_data,
})
loop_csv = os.path.join(OUTPUT_FOLDER, f"{RUN_LABEL}_hysteresis.csv")
loop_df.to_csv(loop_csv, index=False)

snapshot_df = pd.DataFrame(snapshot_records)
snapshot_csv = os.path.join(OUTPUT_FOLDER, f"{RUN_LABEL}_snapshot_manifest.csv")
snapshot_df.to_csv(snapshot_csv, index=False)

print(f"Loop CSV: {loop_csv}")
print(f"Snapshot manifest: {snapshot_csv}")
print(f"Saved {len(snapshot_df)} state files.")
snapshot_df.head()


## Compose a slide-style summary figure

This is a first-pass scientific layout: central hysteresis loop plus the real saved XZ maps. The annotations can be made more ambitious later without rerunning the simulation.

In [ ]:
def plot_npz_snapshot(ax, npz_path, title):
    data = np.load(npz_path, allow_pickle=True)
    x_um = data["x_um"]
    z_um = data["z_um"]
    mz_xz = data["mz_over_Ms"]
    image = ax.imshow(
        mz_xz,
        origin="lower",
        extent=[x_um.min(), x_um.max(), z_um.min(), z_um.max()],
        cmap="coolwarm",
        vmin=-1,
        vmax=1,
        interpolation="nearest",
        aspect="equal",
    )
    ax.set_title(title, fontsize=8, fontweight="bold")
    ax.set_xlabel("x (um)", fontsize=7)
    ax.set_ylabel("z (um)", fontsize=7)
    ax.tick_params(labelsize=7)
    return image


preferred_order = [
    "Positive saturation",
    "Remanence descending",
    "Coercivity descending",
    "Negative saturation",
    "Remanence ascending",
    "Coercivity ascending",
]

available = snapshot_df.set_index("label", drop=False)
ordered = [available.loc[label] for label in preferred_order if label in available.index]

fig = plt.figure(figsize=(14, 8), constrained_layout=True)
gs = GridSpec(3, 4, figure=fig)

ax_loop = fig.add_subplot(gs[:, 1:3])
ax_loop.plot(loop_df["B_ext_T"], loop_df["Mz_over_Ms"], color="#26343a", lw=2.2)
ax_loop.axhline(0, color="0.55", lw=0.8)
ax_loop.axvline(0, color="0.55", lw=0.8)
ax_loop.set_xlabel("B_ext (T)")
ax_loop.set_ylabel("M_z / M_s")
ax_loop.set_title("Magnetization states along the loop", fontsize=16, fontweight="bold")
ax_loop.grid(alpha=0.25)

positions = [gs[0, 0], gs[1, 0], gs[2, 0], gs[0, 3], gs[1, 3], gs[2, 3]]
last_image = None
for row, pos in zip(ordered, positions):
    ax = fig.add_subplot(pos)
    title = f"{row['label']}\nB={row['B_ext_T']:+.2f} T, Mz/Ms={row['Mz_over_Ms']:+.2f}"
    last_image = plot_npz_snapshot(ax, row["npz_path"], title)
    ax_loop.scatter(row["B_ext_T"], row["Mz_over_Ms"], s=55, zorder=5)

if last_image is not None:
    cbar = fig.colorbar(last_image, ax=fig.axes, fraction=0.018, pad=0.01)
    cbar.set_label("m_z / M_s")

summary_png = os.path.join(OUTPUT_FOLDER, f"{RUN_LABEL}_snapshot_summary.png")
summary_svg = os.path.join(OUTPUT_FOLDER, f"{RUN_LABEL}_snapshot_summary.svg")
fig.savefig(summary_png, dpi=300)
fig.savefig(summary_svg)
plt.show()

print(f"Summary PNG: {summary_png}")
print(f"Summary SVG: {summary_svg}")


## Render one selected state

Use this final cell when you want a publication/presentation file for one state. Select by `STATE_INDEX` from the manifest or by giving `STATE_FILE` as the path to one saved `.npz` state file. The cell exports both PNG and SVG.


In [ ]:
STATE_INDEX = 0       # row number in snapshot_df / step index to render
STATE_FILE = r""      # optional: explicit path to a saved state_*.npz file
RENDER_TAG = "selected_state"

if STATE_FILE:
    selected_npz = STATE_FILE
else:
    if "snapshot_df" not in globals():
        raise RuntimeError("Run the simulation cell first, or set STATE_FILE to an existing state_*.npz path.")
    row = snapshot_df.loc[snapshot_df["step_index"] == STATE_INDEX]
    if row.empty:
        raise ValueError(f"No state with step_index={STATE_INDEX}. Available range: {snapshot_df['step_index'].min()} to {snapshot_df['step_index'].max()}.")
    selected_npz = row.iloc[0]["npz_path"]

basename = f"{RENDER_TAG}_{os.path.splitext(os.path.basename(selected_npz))[0]}"
png_path, svg_path = render_xz_state(selected_npz, outdir=OUTPUT_FOLDER, basename=basename, show=True)
print(f"PNG saved: {png_path}")
print(f"SVG saved: {svg_path}")


## Select folder for interactive plot

Use this cell to build an interactive Plotly/HTML viewer from a folder of saved `state_*.npz` files. It does not rerun OOMMF. If `INTERACTIVE_RESULTS_FOLDER` is empty, it uses the current `OUTPUT_FOLDER` when available; otherwise it opens a folder picker if `USE_FOLDER_PICKER=True`.


In [ ]:
INTERACTIVE_RESULTS_FOLDER = r""  # e.g. r"C:\Users\vazquez\FunMaP\simulations\Snapshot_Study_..."
USE_FOLDER_PICKER = False
EXPORT_INTERACTIVE_HTML = True


def choose_results_folder():
    if INTERACTIVE_RESULTS_FOLDER:
        return INTERACTIVE_RESULTS_FOLDER
    if "OUTPUT_FOLDER" in globals() and os.path.isdir(OUTPUT_FOLDER):
        return OUTPUT_FOLDER
    if USE_FOLDER_PICKER:
        import tkinter as tk
        from tkinter import filedialog
        root = tk.Tk()
        root.withdraw()
        folder = filedialog.askdirectory(title="Select snapshot result folder")
        root.destroy()
        if folder:
            return folder
    raise RuntimeError("Set INTERACTIVE_RESULTS_FOLDER to a snapshot output folder, or enable USE_FOLDER_PICKER.")


interactive_folder = choose_results_folder()
state_files = sorted(glob.glob(os.path.join(interactive_folder, "state_*.npz")))
if not state_files:
    state_files = sorted(glob.glob(os.path.join(interactive_folder, "*.npz")))
if not state_files:
    raise FileNotFoundError(f"No .npz state files found in {interactive_folder}")

records = []
for p in state_files:
    d = np.load(p, allow_pickle=True)
    records.append({
        "path": p,
        "step_index": int(d["step_index"]),
        "B_ext_T": float(d["B_ext_T"]),
        "Mz_over_Ms": float(d["Mz_over_Ms"]),
        "label": str(d["label"]),
        "has_xy": "xy_mz_over_Ms" in d.files,
    })
state_table = pd.DataFrame(records).sort_values("step_index").reset_index(drop=True)
state_files = state_table["path"].tolist()

B = state_table["B_ext_T"].to_numpy()
M = state_table["Mz_over_Ms"].to_numpy()
first = np.load(state_files[0], allow_pickle=True)
x = first["x_um"]
z = first["z_um"]
y = first["y_um"] if "y_um" in first.files else x
has_xy = bool(state_table["has_xy"].all())

xz_stack = []
xy_stack = []
for p in state_files:
    d = np.load(p, allow_pickle=True)
    xz_stack.append(d["mz_over_Ms"])
    if has_xy:
        xy_stack.append(d["xy_mz_over_Ms"])
    else:
        xy_stack.append(np.full((len(y), len(x)), np.nan))
xz_stack = np.asarray(xz_stack)
xy_stack = np.asarray(xy_stack)


def state_title(i):
    return f"Step {state_table.loc[i, 'step_index']} | B = {B[i]:+.2f} T | Mz/Ms = {M[i]:+.3f}"


def make_folder_viewer(initial=0):
    fig = make_subplots(
        rows=2,
        cols=2,
        specs=[[{"rowspan": 2}, {}], [None, {}]],
        column_widths=[0.52, 0.48],
        subplot_titles=("Hysteresis loop", "XZ side view", "XY top view" if has_xy else "XY top view not saved"),
        horizontal_spacing=0.10,
        vertical_spacing=0.12,
    )

    fig.add_trace(go.Scatter(
        x=B, y=M, mode="lines+markers",
        line=dict(color="#26343a", width=2),
        marker=dict(size=7, color=state_table["step_index"], colorscale="Viridis", showscale=False),
        customdata=state_table["step_index"],
        hovertemplate="step %{customdata}<br>B=%{x:.2f} T<br>Mz/Ms=%{y:.3f}<extra></extra>"), row=1, col=1)
    fig.add_trace(go.Scatter(
        x=[B[initial]], y=[M[initial]], mode="markers",
        marker=dict(size=16, color="crimson", line=dict(color="white", width=2)),
        hoverinfo="skip"), row=1, col=1)
    fig.add_trace(go.Heatmap(
        x=x, y=z, z=xz_stack[initial], zmin=-1, zmax=1, colorscale="RdBu_r",
        colorbar=dict(title="m_z/M_s", len=0.82)), row=1, col=2)
    fig.add_trace(go.Heatmap(
        x=x, y=y, z=xy_stack[initial], zmin=-1, zmax=1, colorscale="RdBu_r", showscale=False), row=2, col=2)

    frames = []
    for i in range(len(B)):
        frames.append(go.Frame(
            data=[
                go.Scatter(x=B, y=M),
                go.Scatter(x=[B[i]], y=[M[i]]),
                go.Heatmap(x=x, y=z, z=xz_stack[i], zmin=-1, zmax=1, colorscale="RdBu_r"),
                go.Heatmap(x=x, y=y, z=xy_stack[i], zmin=-1, zmax=1, colorscale="RdBu_r"),
            ],
            traces=[0, 1, 2, 3],
            name=str(i),
            layout=go.Layout(title=state_title(i)),
        ))
    fig.frames = frames

    steps = [{
        "label": str(int(state_table.loc[i, "step_index"])),
        "method": "animate",
        "args": [[str(i)], {"mode": "immediate", "frame": {"duration": 0, "redraw": True}, "transition": {"duration": 0}}],
    } for i in range(len(B))]

    fig.update_xaxes(title_text="B_ext (T)", zeroline=True, row=1, col=1)
    fig.update_yaxes(title_text="M_z / M_s", zeroline=True, range=[-1.05, 1.05], row=1, col=1)
    fig.update_xaxes(title_text="x (um)", scaleanchor="y2", scaleratio=1, row=1, col=2)
    fig.update_yaxes(title_text="z (um)", row=1, col=2)
    fig.update_xaxes(title_text="x (um)", scaleanchor="y3", scaleratio=1, row=2, col=2)
    fig.update_yaxes(title_text="y (um)", row=2, col=2)
    fig.update_layout(
        title=state_title(initial), height=820, width=1150, template="plotly_white",
        margin=dict(l=55, r=35, t=80, b=150), showlegend=False,
        sliders=[{"active": initial, "currentvalue": {"prefix": "Step "}, "pad": {"t": 45, "b": 20}, "steps": steps}],
        updatemenus=[{
            "type": "buttons", "direction": "left", "showactive": False,
            "x": 1.0, "y": -0.20, "xanchor": "right", "yanchor": "top",
            "buttons": [
                {"label": "Play", "method": "animate", "args": [None, {"frame": {"duration": 300, "redraw": True}, "transition": {"duration": 0}, "fromcurrent": True}]},
                {"label": "Pause", "method": "animate", "args": [[None], {"frame": {"duration": 0, "redraw": False}, "mode": "immediate", "transition": {"duration": 0}}]},
            ],
        }],
    )
    return fig


interactive_fig = make_folder_viewer(initial=0)
if EXPORT_INTERACTIVE_HTML:
    html_path = os.path.join(interactive_folder, "interactive_snapshot_viewer.html")
    interactive_fig.write_html(html_path, include_plotlyjs="cdn", auto_play=False)
    print(f"HTML saved: {html_path}")

print(f"Loaded {len(state_table)} states from: {interactive_folder}")
interactive_fig


## Cleanup

Run this after the plots are saved if you want to release the OOMMF resources from the notebook session.

In [ ]:
oc.delete(system)
print("OOMMF resources released.")
